# Preprocessing (v2) first step of pipeline

Generates `datasets/KurdiSent_preprocessed.csv` from `datasets/KurdiSent.csv` using
KLPT (Sorani/Arabic): surface and lemma views, Schwa whitelist,
and surface fallback for OOV items.

Runs **before** `build_split_v2`.

**Unchanged:** `SCHWA_FIXES`, `data_cleaning_ckb`, tokenization,
MWE splitting, OOV fallback.

In [1]:
!pip install -q klpt transformers sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 651.7/651.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.6/930.6 kB 15.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ===== [V2-CHANGE 1] Paths from config.py =====
import sys, os, json, hashlib, re, datetime
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/kusa",
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the project "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/kusa'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *

import pandas as pd

print("Input  :", ORIGINAL)
print("Output :", PREPROCESSED)
assert os.path.exists(ORIGINAL), f"KurdiSent.csv missing: {ORIGINAL}"

# ===== [V2-CHANGE 2] Log KLPT version =====
try:
    import importlib.metadata as im
    KLPT_VERSION = im.version("klpt")
except Exception:
    KLPT_VERSION = "unknown"
print("KLPT Version:", KLPT_VERSION)

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

Mounted at /content/drive
project root: /content/drive/MyDrive/kusa
Input  : /content/drive/MyDrive/kusa/datasets/KurdiSent.csv
Output : /content/drive/MyDrive/kusa/datasets/KurdiSent_preprocessed.csv
KLPT Version: 0.1.7


In [3]:
# ===== [V2-CHANGE 3] Safety check =====
FORCE = False

if os.path.exists(MANIFEST) and not FORCE:
    with open(MANIFEST, encoding="utf-8") as f:
        MAN = json.load(f)
    print("A split has already been created:")
    print("  Dev/Test        :", MAN["n_dev"], "/", MAN["n_test"])
    print("  Source SHA-256  :", MAN["source_sha256"])
    if os.path.exists(PREPROCESSED):
        print("  Current file    :", sha256(PREPROCESSED))
    raise SystemExit(
        "Aborted. A new preprocessing run would modify the source of the existing "
        "split. To deliberately reset, first reset build_split_v2 "
        "then set FORCE = True here.")

if os.path.exists(PREPROCESSED) and not FORCE:
    print("Notice: PREPROCESSED already exists and will be overwritten.")
    print("  Previous:", sha256(PREPROCESSED))
print("No split present - proceeding.")

No split present - proceeding.


In [4]:
from klpt.preprocess import Preprocess
from klpt.tokenize import Tokenize
from klpt.stem import Stem

klpt_preprocessor = Preprocess("Sorani", "Arabic")
klpt_tokenizer    = Tokenize("Sorani", "Arabic")
klpt_stemmer      = Stem("Sorani", "Arabic")
print("KLPT loaded.")

KLPT loaded.


In [5]:
# Unchanged from original.
SCHWA_FIXES = {
    "\u0644\u0647": "\u0644\u06d5",   # le  (preposition)
    "\u0628\u0647": "\u0628\u06d5",   # be  (preposition)
    "\u06a9\u0647": "\u06a9\u06d5",   # ke  (relativizer)
    "\u0646\u0647": "\u0646\u06d5",   # ne  (negation)
    "\u0648\u0647": "\u0648\u06d5",   # we  (particle)
}

def normalize_schwa(token):
    return SCHWA_FIXES.get(token, token)

MWE_SEP    = "\u2012"   # KLPT-internal separator in compound tokens
WORD_BOUND = "\u2581"   # KLPT word boundary marker


def data_cleaning_ckb(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    normalized = klpt_preprocessor.preprocess(text).strip()

    # Surface view: natural text, but with the same word-level Schwa correction
    # to maintain token alignment between both views.
    surface = " ".join(normalize_schwa(t) for t in normalized.split())

    tokens = klpt_tokenizer.word_tokenize(normalized)

    lemmatized_words = []
    for word in tokens:
        clean_word = word.replace(WORD_BOUND, "")
        for part in clean_word.split(MWE_SEP):
            part = part.strip()
            if not part:
                continue
            part = normalize_schwa(part)
            try:
                analysis = klpt_stemmer.analyze(part)
                if analysis:
                    lemma_word = analysis[0].get("lemma", part)
                    if isinstance(lemma_word, list):
                        lemma_word = lemma_word[0] if lemma_word else part
                    lemmatized_words.append(str(lemma_word))
                else:
                    lemmatized_words.append(part)    # OOV -> Surface fallback
            except Exception:
                lemmatized_words.append(part)

    return surface, " ".join(lemmatized_words)


# Quick function test
_s, _l = data_cleaning_ckb("\u0628\u0647 \u062e\u06ce\u0631 \u0628\u06ce\u062a 123 http://x.com")
print("Surface:", _s)
print("Lemma  :", _l)

Surface: بە خێر بێت
Lemma  : بوون خێر بوون


In [6]:
print("Loading dataset ...")
df_ckb = pd.read_csv(ORIGINAL)
df_ckb = df_ckb.loc[:, ~df_ckb.columns.str.contains("^Unnamed")]
if "num_label" in df_ckb.columns:
    df_ckb["label"] = df_ckb["num_label"]
n_raw = len(df_ckb)
print(f"{n_raw} rows, columns: {list(df_ckb.columns)}")

print("\nPreprocessing in progress (takes a few minutes) ...")
df_ckb[["surface", "lemma"]] = df_ckb["text"].apply(
    lambda x: pd.Series(data_cleaning_ckb(str(x)))
)
df_ckb_clean = df_ckb[df_ckb["surface"].str.strip() != ""].reset_index(drop=True)

print(f"\n{n_raw} loaded, {len(df_ckb_clean)} kept "
      f"({n_raw - len(df_ckb_clean)} discarded with empty surface)")
print("Label distribution:", df_ckb_clean["label"].value_counts().sort_index().to_dict())
print("Categories        :", df_ckb_clean["category"].value_counts().to_dict())

df_ckb_clean.to_csv(PREPROCESSED, index=False, encoding="utf-8")
print("\nSaved:", PREPROCESSED)

Loading dataset ...
12306 rows, columns: ['num_label', 'category', 'text', 'label']

Preprocessing in progress (takes a few minutes) ...

12306 loaded, 12306 kept (0 discarded with empty surface)
Label distribution: {0: 4102, 1: 4102, 2: 4102}
Categories        : {'social': 5463, 'news': 4302, 'art': 1759, 'health': 624, 'technology': 158}

Saved: /content/drive/MyDrive/kusa/datasets/KurdiSent_preprocessed.csv


In [7]:
# ===== [V2-CHANGE 4] Metadata for traceability =====
n_lemma_eq_surface = int((df_ckb_clean["lemma"].astype(str).str.strip()
                          == df_ckb_clean["surface"].astype(str).str.strip()).sum())

meta = {
    "created":        datetime.datetime.now().isoformat(timespec="seconds"),
    "klpt_version":   KLPT_VERSION,
    "source":         ORIGINAL,
    "source_sha256":  sha256(ORIGINAL),
    "output":         PREPROCESSED,
    "output_sha256":  sha256(PREPROCESSED),
    "n_rows_in":      int(n_raw),
    "n_rows_out":     int(len(df_ckb_clean)),
    "n_dropped_empty_surface": int(n_raw - len(df_ckb_clean)),
    "n_lemma_identical_to_surface": n_lemma_eq_surface,
    "schwa_whitelist_size": len(SCHWA_FIXES),
}
with open(os.path.join(DATA, "preprocessing_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(json.dumps(meta, indent=2, ensure_ascii=False))
print("\nNext step: build_split_v2")

{
  "created": "2026-08-11T00:20:54",
  "klpt_version": "0.1.7",
  "source": "/content/drive/MyDrive/kusa/datasets/KurdiSent.csv",
  "source_sha256": "fe90d301bc431495fb5002b0e1651f2f8287eb641dd446072f2f4ea5ccde7fc0",
  "output": "/content/drive/MyDrive/kusa/datasets/KurdiSent_preprocessed.csv",
  "output_sha256": "1ed2d675068f4d524441822b0e392596f1349e2bb9c56c65aef082cfedcdc2b6",
  "n_rows_in": 12306,
  "n_rows_out": 12306,
  "n_dropped_empty_surface": 0,
  "n_lemma_identical_to_surface": 12,
  "schwa_whitelist_size": 5
}

Next step: build_split_v2
